# Elevated Bloom Risk Prediction — Train All Three Models

This notebook trains all three baseline learners (Logistic Regression, Random Forest, XGBoost) on the Kerala + Karnataka weekly dataset.

**Important scope note:** the target variable is *elevated bloom risk* — operationally defined as satellite-derived chlorophyll-a > 2 mg/m³ per 8-day window, combined with CMFRI-documented events. This is a proxy for bloom conditions, **not** confirmed toxic HAB events. See `LIMITATIONS.md` at the repo root for the four honest caveats (n=4 CMFRI closure ceiling, Trichodesmium species blind spot, bidirectional reliability errors, 5-year window).

Runs end-to-end in **Google Colab**:
1. Clones the repo (if not already present).
2. Installs dependencies.
3. Trains Logistic Regression, Random Forest, and XGBoost — each in its own cell.
4. Shows metrics + feature importances.

Open this notebook in Colab, then hit **Runtime → Run all**.

## 1. Setup (Colab only)

**If you're running locally after `git clone`, skip this cell.**

In [ ]:
# --- Colab: clone repo + install deps (safe to re-run) ---
import os, sys, pathlib
REPO = "BloomWatch_AI"
REPO_URL = "https://github.com/AvniCat/BloomWatch_AI.git"
if not pathlib.Path(REPO).exists():
    !git clone $REPO_URL
os.chdir(REPO)
!pip install -q -r requirements.txt
sys.path.insert(0, str(pathlib.Path.cwd() / "code" / "src"))
print("cwd:", os.getcwd())
print("sys.path[0]:", sys.path[0])

## 2. Peek at the data

In [ ]:
import pandas as pd
df = pd.read_csv("data/dataset_merged_india_monthly_wide.csv")
print("shape:", df.shape)
df.head()

In [ ]:
from feature_prep import build_features
features, train, test = build_features()
print(f"features: {len(features)}")
print(f"train: {len(train)} rows  ({train['bloom'].mean():.3f} bloom rate)")
print(f"test:  {len(test)} rows  ({test['bloom'].mean():.3f} bloom rate)")

## 3. Train — Logistic Regression

In [ ]:
from train_logistic_regression import main as train_lr
train_lr()

## 4. Train — Random Forest

In [ ]:
from train_random_forest import main as train_rf
train_rf()

## 5. Train — XGBoost

In [ ]:
from train_xgboost import main as train_xgb
train_xgb()

## 6. Compare metrics side-by-side

In [ ]:
import pandas as pd
pd.read_csv("results/model_metrics.csv")

## 7. Top drivers per model (mitigation levers)

In [ ]:
for name in ["LogisticRegression", "RandomForest", "XGBoost"]:
    print(f"\n=== {name} — top 10 features ===")
    imp = pd.read_csv(f"results/feature_importance/{name}_feature_importance.csv").head(10)
    print(imp.to_string(index=False))

## 8. Load a saved model and predict on new data

In [ ]:
import pickle
with open("results/models/RandomForest.pkl", "rb") as f:
    rf = pickle.load(f)

# Example: run the trained model on the test set (same features)
features, train, test = build_features()
X_test = test[features]
probs = rf.predict_proba(X_test)[:, 1]
print("first 5 predicted bloom probabilities:", probs[:5].round(3))